In [ ]:
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Setup directory paths (using current working directory for Jupyter)
notebook_dir = os.getcwd()
root_dir=os.path.dirname(os.path.dirname(notebook_dir))

output_dir = os.path.join(root_dir, "output", "sh-03")
file_path = os.path.join(root_dir, "data", "SH-03 COBBLE 11122024.parquet")

print(f"Target file path: {file_path}")
print(f"Output directory: {output_dir}")

In [ ]:
print(f"Reading and downsampling data from {file_path}")
print("Processing in chunks with vectorized downsampling...")

parquet_file = pq.ParquetFile(file_path)
downsampled_chunks = []
global_row_count = 0

# Process in batches of 10,000 rows
for batch in parquet_file.iter_batches(batch_size=10000):
    df_chunk = batch.to_pandas()
    chunk_len = len(df_chunk)
    
    # Calculate the exact starting index for this chunk to maintain the global "every 100th row" rule
    remainder = global_row_count % 100
    start_idx = 0 if remainder == 0 else (100 - remainder)
    
    if start_idx < chunk_len:
        # Slicing with [start::100] is fully vectorized and incredibly fast
        downsampled_chunks.append(df_chunk.iloc[start_idx::100].copy())
        
    global_row_count += chunk_len
    print(f"Processed {global_row_count} rows...", end='\r')
    
print(f"\nFinished reading. Concatenating chunks...")
df_down = pd.concat(downsampled_chunks, ignore_index=True)
print(f"Downsampled dataframe shape: {df_down.shape}")

# Ensure Timestamp is handled correctly
if 'Timestamp' in df_down.columns:
    df_down['Timestamp'] = pd.to_datetime(df_down['Timestamp'])
    print("Timestamp column successfully converted to datetime.")

In [ ]:
print("Pre-calculating standard deviations for all columns...")
# Convert bools to float for unified std calculation; non-numeric columns will be ignored here
numeric_bool_df = df_down.select_dtypes(include=[np.number, 'bool'])
stds = numeric_bool_df.astype(float).std().fillna(0)
print("Standard deviations calculated successfully.")

In [ ]:
print("Identifying column types (Boolean, Constant, Analog)...")
constant_cols = []
bool_cols = []
analog_cols = []

# Process numeric/boolean columns based on pre-calculated stds
for col in df_down.columns:
    if col in stds.index:
        col_std = stds[col]
        if col_std == 0:
            constant_cols.append(col)
        elif pd.api.types.is_bool_dtype(df_down[col]):
            bool_cols.append(col)
        else:
            # Fast check for 0/1 columns using downsampled data
            unique_vals = set(df_down[col].dropna().unique())
            if unique_vals.issubset({0, 1, 0.0, 1.0}):
                bool_cols.append(col)
            else:
                analog_cols.append(col)
    else:
        # Handle non-numeric columns (e.g., strings/objects)
        if df_down[col].nunique(dropna=False) <= 1:
            constant_cols.append(col)
        else:
            analog_cols.append(col)

print(f"Found {len(bool_cols)} boolean, {len(constant_cols)} constant, and {len(analog_cols)} analog columns.")

In [ ]:
def sort_cols_by_std(cols):
    col_stds = {c: stds[c] if c in stds.index else 0 for c in cols}
    return sorted(col_stds, key=col_stds.get, reverse=True)

print("Sorting columns by standard deviation (high fluctuation to low)...")
sorted_bool = sort_cols_by_std(bool_cols)
sorted_analog = sort_cols_by_std(analog_cols)
sorted_const = sort_cols_by_std(constant_cols)

print("Saving categorized datasets to CSV...")
# Updated saving paths to output_dir
df_down[sorted_bool].to_csv(os.path.join(output_dir, "boolean_columns.csv"), index=False)
df_down[sorted_analog].to_csv(os.path.join(output_dir, "analog_columns.csv"), index=False)
df_down[sorted_const].to_csv(os.path.join(output_dir, "constant_columns.csv"), index=False)
print(f"Categorized CSVs saved successfully to {output_dir}")

In [ ]:
print("Extracting specific parameters (Speed, Torque, Current, Power, Looper, Vibration)...")
param_keywords = {
    'speed': ['speed'],
    'torque': ['torque'],
    'current': ['current'],
    'power': ['power'],
    'looper': ['loop', 'l1112', 'l1213', 'l1314', 'l1415', 'l1516', 'l1617', 'l1718', 'l1819', 'l1920', 'l2021', 'l24fb'],
    'vibration': ['vib'] # Updated keyword from 'vibr' to 'vib'
}

params_cols = {k: [] for k in param_keywords.keys()}
all_cols_lower = {str(col).lower(): col for col in df_down.columns}

# Vector-like matching using dictionary lookups
for col_lower, original_col in all_cols_lower.items():
    for param, keywords in param_keywords.items():
        if any(kw in col_lower for kw in keywords):
            params_cols[param].append(original_col)

all_param_cols = list(set().union(*params_cols.values()))

# Add Timestamp to the front of all_parameters if it exists
if 'Timestamp' in df_down.columns and 'Timestamp' not in all_param_cols:
    all_param_cols.insert(0, 'Timestamp')

print(f"Found {len(all_param_cols)} columns across all specified parameters.")
df_down[all_param_cols].to_csv(os.path.join(output_dir, "all_parameters.csv"), index=False)

In [ ]:
print("Finding highest standard deviation column for each parameter...")
highest_cols = []
for param, cols in params_cols.items():
    if not cols:
        print(f"  {param}: No columns found.")
        continue
    
    param_stds = {c: stds[c] if c in stds.index else 0 for c in cols}
    highest_col = max(param_stds, key=param_stds.get)
    highest_val = param_stds[highest_col]
    
    print(f"  {param}: {highest_col} (std = {highest_val:.4f})")
    highest_cols.append(highest_col)

# Ensure Timestamp accompanies highest columns output
output_cols = list(highest_cols)
if 'Timestamp' in df_down.columns:
    output_cols.insert(0, 'Timestamp')

print("\nSaving highest fluctuation dataset...")
df_down[output_cols].to_csv(os.path.join(output_dir, "highest_std_parameters.csv"), index=False)